# SnapKnow — ML starter notebook

Loads `snapknow_ml_dataset.csv` (made by `build_ml_dataset.py`) and walks through:
1. Exploring the data
2. Predicting **category** from simple **image features** (colour, brightness, edges...)
3. Predicting **category** from the **AI-written text** (TF-IDF)
4. Unsupervised clustering of the images

> ⚠️ **Dataset size:** the CSV built from your saved history is tiny (a handful of rows), so any score
> here is only a demo of the *pipeline*, not a trustworthy result. To get a real dataset, label a folder of
> images in bulk: `python build_ml_dataset.py --folder my_images --provider gemini` (aim for 50+ images per class).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("snapknow_ml_dataset.csv")
print(df.shape)
df.head()

## 1. Explore

In [ ]:
print(df.isna().sum()[lambda s: s > 0])          # columns with missing values
print()
print(df["category"].value_counts())

df["category"].value_counts().plot(kind="bar", title="Images per category", rot=0)
plt.ylabel("count"); plt.show()

In [ ]:
LABEL_AND_TEXT = ["image_id", "image_filename", "source", "category", "name", "true_label", "ai_correct",
                  "caption", "description", "details", "fun_fact"]
# only the always-present numeric columns (enrichment columns are optional / sparse)
IMAGE_FEATURES = [c for c in df.columns
                  if c.startswith(("img_", "mean_", "std_", "hue_")) or
                  c in ("brightness", "contrast", "mean_saturation", "colorfulness", "sharpness", "edge_density")]
TEXT_STATS = [c for c in df.columns if c.endswith("_count") and c in df.columns and not c.startswith(("inat", "off", "wikidata"))]

print(len(IMAGE_FEATURES), "image features:", IMAGE_FEATURES)
print(len(TEXT_STATS), "text-stat features:", TEXT_STATS)

df.groupby("category")[["brightness", "mean_saturation", "colorfulness", "edge_density"]].mean().round(3)

In [ ]:
corr = df[IMAGE_FEATURES].corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
plt.colorbar(im); plt.title("Image-feature correlations"); plt.tight_layout(); plt.show()

## 2. Image features → category

Random forest on colour/brightness/edge features, evaluated with cross-validation (never score on training data).
Classes with fewer than 2 images are dropped because they can't be split across folds.

In [ ]:
counts = df["category"].value_counts()
usable = df[df["category"].isin(counts[counts >= 2].index)].reset_index(drop=True)
dropped = sorted(set(df["category"]) - set(usable["category"]))
if dropped:
    print("Dropped classes with <2 samples:", dropped)

n_splits = int(min(5, usable["category"].value_counts().min()))
print(f"{len(usable)} rows, {usable['category'].nunique()} classes, {n_splits}-fold CV")

X, y = usable[IMAGE_FEATURES], usable["category"]
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

rf = RandomForestClassifier(n_estimators=300, random_state=42)
pred = cross_val_predict(rf, X, y, cv=cv)
print(classification_report(y, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y, pred, xticks_rotation=45)
plt.title("Image features -> category"); plt.show()

In [ ]:
rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=IMAGE_FEATURES).sort_values().tail(12)
imp.plot(kind="barh", title="Top image features"); plt.show()

## 3. AI-written text → category

TF-IDF on caption + description + details, then logistic regression. Note: the text was written by the same
AI that chose the label, so this is *easy* (label leakage) — treat it as a text-pipeline demo, not a fair benchmark.
Use `true_label` (when you have it) as the honest target.

In [ ]:
usable["text"] = usable["caption"].fillna("") + " " + usable["description"].fillna("") + " " + usable["details"].fillna("")
text_model = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2)),
                           LogisticRegression(max_iter=1000))
pred_t = cross_val_predict(text_model, usable["text"], usable["category"], cv=cv)
print(classification_report(usable["category"], pred_t, zero_division=0))

## 4. Unsupervised: do images cluster by category?

In [ ]:
Xs = StandardScaler().fit_transform(df[IMAGE_FEATURES])
k = min(df["category"].nunique(), len(df) - 1)
df["cluster"] = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xs)
print(pd.crosstab(df["category"], df["cluster"]))

pts = PCA(n_components=2).fit_transform(Xs)
for cat in df["category"].unique():
    m = (df["category"] == cat).values
    plt.scatter(pts[m, 0], pts[m, 1], label=cat, s=70)
plt.legend(); plt.title("PCA of image features"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

## Next steps
- **Get more data** — `python build_ml_dataset.py --folder my_images` (sub-folder names become `true_label`).
- **Measure the AI itself** — with `true_label`, `ai_correct` shows how often the vision model is right; check whether low `confidence` predicts mistakes (calibration).
- **Add external-API columns** — re-run with `--enrich` for iNaturalist / Open Food Facts / Wikidata / geo features.
- **Better image features** — extract CNN embeddings (e.g. a pretrained ResNet) instead of hand-made colour stats.
- **Predict `confidence`** (regression) or **`details_word_count`** as alternative targets.